This is the 5 qubit t-gate injection circuit

In [1]:
import cirq
import numpy as np

In [2]:

# 4 data qubits + 1 ancilla
q0, q1, q2, q3, q4 = cirq.LineQubit.range(5)

circuit = cirq.Circuit()

# --- Prepare ancilla in |T> = T|+> ---
circuit.append([
    cirq.H(q0),
    cirq.T(q0)
])

# --- T gate injection on data qubits ---
circuit.append(cirq.CNOT(q0, q1))
circuit.append(cirq.CNOT(q0, q2))
circuit.append(cirq.CNOT(q0, q3))


# Measure ancilla
m_anc = cirq.measure(q0, key='m_anc')
circuit.append(m_anc)

# Apply conditional S correction on data qubit
circuit.append(cirq.S(q1).with_classical_controls('m_anc'))
circuit.append(cirq.S(q2).with_classical_controls('m_anc'))
circuit.append(cirq.S(q3).with_classical_controls('m_anc'))


# --- (Optional) Other data qubits remain untouched ---
# q1, q2 are spectators in this example

print(circuit)

                                  ┌───┐
0: ───────H───T───@───@───@───M───────────
                  │   │   │   ║
1: ───────────────X───┼───┼───╫────S──────
                      │   │   ║    ║
2: ───────────────────X───┼───╫────╫S─────
                          │   ║    ║║
3: ───────────────────────X───╫────╫╫S────
                              ║    ║║║
m_anc: ═══════════════════════@════^^^════
                                  └───┘


Without measurement the unitary is hence:

U_{CS} =
\begin{pmatrix}
T_L & S T_L \\
? & ?
\end{pmatrix}

Note thaat "?" does not matter as we start the ancilla in|0>; based on the measurement we then applay T_L or S T_L

In [3]:
circuit = cirq.Circuit()

# --- Prepare ancilla in |T> = T|+> ---
circuit.append([
    cirq.H(q0),
    cirq.T(q0)
])

# --- T gate injection on data qubits ---
circuit.append(cirq.CNOT(q1, q0))
circuit.append(cirq.CNOT(q2, q0))
circuit.append(cirq.CNOT(q3, q0))


U = cirq.unitary(circuit)

threshold = 1e-10
U[np.abs(U) < threshold] = np.nan
np.set_printoptions(precision=3, 
                    linewidth=500, 
                    threshold=1000000, 
                    edgeitems=10000000, 
                    nanstr=None)
print(np.sqrt(2)*U)

[[ 1.   +0.j       nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj  1.   +0.j       nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj]
 [   nan  +nanj  0.707+0.707j    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj -0.707-0.707j    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj]
 [   nan  +nanj    nan  +nanj  0.707+0.707j    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj -0.707-0.707j    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj]
 [   nan  +nanj    nan  +nanj    nan  +nanj  1.   +0.j       nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj  1.   +0.j       nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj]
 [   nan  +nanj    nan  +nanj    nan  +nanj    nan  +nanj  0.707+0.707j    nan  +nanj    nan

In [4]:
#So T_L is
#
# Note: need to renomalise by sqrt 2 for unitary
#
print(np.sqrt(2)*U[:8,:8])

[[1.   +0.j      nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj]
 [  nan  +nanj 0.707+0.707j   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj]
 [  nan  +nanj   nan  +nanj 0.707+0.707j   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj]
 [  nan  +nanj   nan  +nanj   nan  +nanj 1.   +0.j      nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj]
 [  nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj 0.707+0.707j   nan  +nanj   nan  +nanj   nan  +nanj]
 [  nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj 1.   +0.j      nan  +nanj   nan  +nanj]
 [  nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj 1.   +0.j      nan  +nanj]
 [  nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj   nan  +nanj 0.707+0.707j]]


In [5]:
T = np.diag([1, (1+1j)/np.sqrt(2)])

print(T)

[[1.   +0.j    0.   +0.j   ]
 [0.   +0.j    0.707+0.707j]]


In [6]:
#
#   
#
T_alt = np.kron(np.kron(T,T),T)

threshold = 1e-10
T_alt[np.abs(T_alt) < threshold] = np.nan
print(T_alt)

[[ 1.000e+00+0.j           nan+0.j           nan+0.j           nan+0.j           nan+0.j           nan+0.j           nan+0.j           nan+0.j   ]
 [       nan+0.j     7.071e-01+0.707j        nan+0.j           nan+0.j           nan+0.j           nan+0.j           nan+0.j           nan+0.j   ]
 [       nan+0.j           nan+0.j     7.071e-01+0.707j        nan+0.j           nan+0.j           nan+0.j           nan+0.j           nan+0.j   ]
 [       nan+0.j           nan+0.j           nan+0.j     2.237e-17+1.j           nan+0.j           nan+0.j           nan+0.j           nan+0.j   ]
 [       nan+0.j           nan+0.j           nan+0.j           nan+0.j     7.071e-01+0.707j        nan+0.j           nan+0.j           nan+0.j   ]
 [       nan+0.j           nan+0.j           nan+0.j           nan+0.j           nan+0.j     2.237e-17+1.j           nan+0.j           nan+0.j   ]
 [       nan+0.j           nan+0.j           nan+0.j           nan+0.j           nan+0.j           nan+0.j     2.237e-

In [8]:
#
#   T_L != T\otimes T \otimes T\otimes T?
#
print(np.diagonal(np.sqrt(2)*U[:8,:8]))
print(np.diagonal(T_alt))

#
# Not the same
#
print(np.diagonal(np.sqrt(2)*U[:8,:8]) - np.diagonal(T_alt))

#
#   max diff
#
print(np.max(abs(np.diagonal(np.sqrt(2)*U[:8,:8]) - np.diagonal(T_alt))))
tmp = np.argmax(abs(np.diagonal(np.sqrt(2)*U[:8,:8]) - np.diagonal(T_alt)))
print(tmp, np.sqrt(2)*U[tmp,tmp], np.diagonal(T_alt)[tmp])

[1.   +0.j    0.707+0.707j 0.707+0.707j 1.   +0.j    0.707+0.707j 1.   +0.j    1.   +0.j    0.707+0.707j]
[ 1.000e+00+0.j     7.071e-01+0.707j  7.071e-01+0.707j  2.237e-17+1.j     7.071e-01+0.707j  2.237e-17+1.j     2.237e-17+1.j    -7.071e-01+0.707j]
[2.220e-16+0.00e+00j 2.220e-16+1.11e-16j 2.220e-16+1.11e-16j 1.000e+00-1.00e+00j 2.220e-16+1.11e-16j 1.000e+00-1.00e+00j 1.000e+00-1.00e+00j 1.414e+00+2.22e-16j]
1.414213562373095
3 (1.0000000000000002+0j) (2.2371143170757382e-17+0.9999999999999998j)


In [9]:
#
#   TO DO
#
#   Can you rewirte the T injection without ancilla but instead with
#   TL = np.sqrt(2)*U[:16,:16])
#
#   For example use: https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.DiagonalGate
#
TL_diago = np.array([1, (1+1j)/np.sqrt(2), (1+1j)/np.sqrt(2), 1,
            (1+1j)/np.sqrt(2), 1, 1, (1+1j)/np.sqrt(2)])

print(np.max(abs(np.diagonal(np.sqrt(2)*U[:8,:8]) - TL_diago)))

#tmp = np.argmax(abs(np.diagonal(np.sqrt(2)*U[:16,:16]) - TL_diago))
#print(tmp, np.sqrt(2)*U[tmp,tmp], TL_diago[tmp])

2.482534153247273e-16
